<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

# Unsupervised Learning: Discovering Customer Segments in Retail Data
### CP020003 Artificial Intelligence — In-Class Notebook (Full Unsupervised Learning Deep-Dive)

For the last 4 weeks you have worked with **labeled data** — every row came with an answer key (`Class_Label`,
`Disease_Risk`, etc.) and the model's job was to learn the mapping from features to that label. This week, the
answer key disappears. 🕵️

Real businesses almost never start with neat labels like *"this customer is a VIP"* or *"this customer is
about to churn."* They start with a pile of raw transactions and a genuine question: **"What natural groups
exist in our customers, and what should we *do* about them?"** That is precisely the job of **Unsupervised
Learning**.

We will use a real **retail transactions dataset** — 1,000 purchases across Beauty, Clothing, and Electronics
— and treat it as if we just got hired as the first data scientist at this store. By the end of this notebook
you will be able to:

1. Explain *why* unsupervised learning is the right tool when there is no ground-truth label
2. Perform a proper **EDA** on transactional retail data and spot trends before modeling anything
3. Engineer customer-level behavioral features (an RFM-style feature set) from raw transactions
4. Explain **why Feature Scaling (StandardScaler) is not optional** for distance-based clustering
5. Explain the **Curse of Dimensionality** and demonstrate it with your own eyes, in code
6. Use **PCA** both to fight the curse of dimensionality and to visualize clusters in 2D
7. Choose the right **K** using the Elbow Method, Silhouette Score, Davies–Bouldin Index, and
   Calinski–Harabasz Index
8. Understand **K-Means** at the algorithm level — including **Random Init vs. K-Means++**
9. Understand **DBSCAN** at the algorithm level — including the role of `eps` and `min_samples`, and *why*
   it can succeed exactly where K-Means fails
10. Know **when** to reach for K-Means, DBSCAN, or a **deep-learning-based** clustering approach
11. Meet two more essential unsupervised tools: **Hierarchical Clustering** and **Gaussian Mixture Models**
12. Most importantly: turn cluster numbers into **business personas** and concrete, profit-driving **actions**

Runs fully on **Google Colab (free CPU)** — nothing in this notebook needs a GPU, and the whole notebook
executes in well under a minute. ⏱️


## 0. Setup

We need the usual data-science stack (`pandas`, `numpy`, `matplotlib`, `seaborn`) plus `scikit-learn` for every
clustering algorithm, metric, and preprocessing tool we'll use today. Colab already ships with all of these
pre-installed, so this cell just imports them.


In [ ]:
# Run this once per Colab session if a package is ever missing
# !pip -q install scikit-learn --upgrade

import warnings
warnings.filterwarnings("ignore")

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_moons, make_blobs
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110


In [ ]:
RANDOM_STATE =  # Pick your lucky number here
np.random.seed(RANDOM_STATE)

print("Libraries loaded ✅")

## 1. Why Unsupervised Learning? 🤔

| | **Supervised Learning** (Weeks 1–4) | **Unsupervised Learning** (This week) |
|---|---|---|
| **Data** | Features **+ known labels** (`y`) | Features **only** — no `y` at all |
| **Goal** | Predict a known target as accurately as possible | Discover hidden **structure**, **groups**, or **patterns** |
| **Question it answers** | *"Is this transaction fraud or not?"* | *"What **kinds** of customers do we even have?"* |
| **Typical retail use** | Predicting churn, predicting demand, predicting price sensitivity — **if you already have historical labels** | Customer segmentation, anomaly/fraud detection, market-basket grouping, dimensionality reduction — **when nobody has labeled anything yet** |
| **Success looks like** | High accuracy / F1 on held-out labels | Groups that are **statistically coherent** *and* **make business sense** |

### The retail reality 🏬

No retailer's database has a column called `Customer_Segment`. Nobody sat down and manually tagged 1,000
transactions as *"Budget Shopper"* or *"VIP."* Instead, the business only has **raw transactions**: who
bought what, when, how much they spent, and how old they are.

Unsupervised learning lets us **discover** the groups that are already hiding in that data — and then, ​
critically, translate each group into a **story and an action**. That last step — going from *"Cluster 2 has
centroid [0.8, -0.3, 1.2]"* to *"these are our highest-value loyal shoppers, protect them with a VIP
program"* — is the single most important (and most often skipped!) skill in applied unsupervised learning.
We will practice it heavily today.


## 2. Load the Dataset 📥

**Dataset:** Retail Sales Transactions — 1,000 rows, 9 columns (Transaction ID, Date, Customer ID, Gender,
Age, Product Category, Quantity, Price per Unit, Total Amount).

**Credit:** Originally published on Kaggle by *mohammadtalib786* —
[Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset). We load our
class copy directly from GitHub so everyone works with the exact same file.

We wrap the download in a `try/except` — if for any reason the network call fails (e.g. a flaky Colab
session), we fall back to a small embedded sample so the rest of the notebook still runs end-to-end.


In [ ]:
# ==========================
# TODO: Student: Enter your dataset name
# ==========================
YOUR_DATASET_NAME = " # Write your dataset name here "

url = (
    f"https://raw.githubusercontent.com/"
    f"kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/main/dataset/"
    f"{YOUR_DATASET_NAME}.csv"
)

df = pd.read_csv(url)
df["Date"] = pd.to_datetime(df["Date"])

df.head()

In [ ]:
print("Shape:", df.shape)
print()
df.info()
print()
df.describe(include="all").T


## 3. Exploratory Data Analysis — Understanding Trends First 📊

Before we cluster anything, we need to actually *look* at the data. Good EDA answers three questions:
1. What does a "typical" transaction/customer look like?
2. Are there obvious trends over time, category, or demographics?
3. Are any features skewed, correlated, or on wildly different scales? (This directly foreshadows Section 5!)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["Age"], bins=20, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Age Distribution")

sns.countplot(data=df, x="Gender", ax=axes[1], palette="pastel")
axes[1].set_title("Gender Split")

sns.countplot(data=df, x="Product Category", ax=axes[2],
              order=df["Product Category"].value_counts().index, palette="Set2")
axes[2].set_title("Transactions per Category")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Total Amount"], bins=30, kde=True, ax=axes[0], color="#55A868")
axes[0].set_title("Total Amount Distribution (per transaction)")
axes[0].set_xlabel("Total Amount ($)")

sns.boxplot(data=df, x="Product Category", y="Total Amount", ax=axes[1], palette="Set2")
axes[1].set_title("Spending Range by Category")

plt.tight_layout()
plt.show()

print(df.groupby("Product Category")["Total Amount"].agg(["mean", "median", "count"]).round(1))


**Trend #1 — Electronics is a "high-roller" category.** Electronics has the widest spread and highest
median spend (expensive individual items, e.g. $500 unit price), while Beauty is a low-ticket, high-frequency
category. Clothing sits in between. This alone hints that *"average spend"* will be a very useful clustering
feature.


In [ ]:
monthly = df.set_index("Date").resample("ME")["Total Amount"].agg(["sum", "count"])
monthly.columns = ["Revenue", "Num Transactions"]

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(monthly.index, monthly["Revenue"], marker="o", color="#C44E52", label="Revenue")
ax1.set_ylabel("Monthly Revenue ($)", color="#C44E52")
ax1.set_xlabel("Month")

ax2 = ax1.twinx()
ax2.bar(monthly.index, monthly["Num Transactions"], alpha=0.25, width=15,
        color="#4C72B0", label="# Transactions")
ax2.set_ylabel("# Transactions", color="#4C72B0")

plt.title("Monthly Revenue & Transaction Volume Trend")
fig.tight_layout()
plt.show()


**Trend #2 — Revenue and transaction volume don't always move together.** A month can have *more*
transactions but *lower* revenue (lots of small Beauty purchases) or *fewer* transactions but *higher*
revenue (a handful of big Electronics purchases). This is exactly why looking at revenue alone, or count
alone, would mislead a retailer — you need **both**, which is the whole idea behind RFM-style features
(Section 4).


In [ ]:
numeric_cols =  # Write your code here
corr = df[numeric_cols].corr()

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Heatmap (numeric features)")
plt.show()

pd.crosstab(df["Product Category"], df["Gender"], normalize="index").round(2)


**Trend #3 — `Total Amount` is basically `Quantity × Price per Unit`** (correlation close to 1 with
Price per Unit is expected), while `Age` is essentially uncorrelated with spending. That last point is a
useful, slightly counter-intuitive business insight on its own: **age is not a great proxy for value in this
store** — we shouldn't assume "older = bigger spender." Clustering will help us find what *actually* separates
high-value customers from low-value ones.


## 4. Feature Engineering — Building a Customer-Level Feature Set 🧱

Clustering algorithms don't cluster "transactions" or "rows" in the abstract — they cluster **whatever
feature vector you hand them**. Choosing that feature vector well is 80% of the job.

### A quick, honest data note 🔍

The classic **RFM** framework (**R**ecency, **F**requency, **M**onetary) segments customers using:
- **Recency** — days since their last purchase
- **Frequency** — how many times they've purchased
- **Monetary** — how much they've spent in total

In *this particular dataset*, every `Customer ID` appears **exactly once** — there are no repeat purchases
to aggregate. That means true `Frequency` (transactions per customer) is 1 for everybody, so we can't build
the full classic RFM here. **This is a real, common data limitation — not something to hide from students.**
In a production dataset with repeat customers, you would compute:

```python
rfm = df.groupby("Customer ID").agg(
    Recency=("Date", lambda d: (reference_date - d.max()).days),
    Frequency=("Transaction ID", "count"),
    Monetary=("Total Amount", "sum"),
)
```

Since we only have one transaction per customer here, we build the closest honest equivalent — a
**behavioral feature set** using what this single transaction *does* tell us: how recently it happened, how
much was spent, how large the basket was, what was bought, and who the shopper is. This still lets us
practice every technique (scaling, PCA, K selection, K-Means, DBSCAN) exactly as we would on real RFM data.


In [ ]:
reference_date =  # Write your code here

features = pd.DataFrame(index=df.index)
features["Recency"] =  # Write your code here
features["Monetary"] =  # Write your code here
features["Quantity"] =  # Write your code here
features["AvgItemPrice"] = df["Price per Unit"]
features["Age"] = df["Age"]

# One-hot encode Product Category and Gender so the model can use them too
features = pd.concat(
    [features, pd.get_dummies(df["Product Category"], prefix="Cat"),
     pd.get_dummies(df["Gender"], prefix="Gender")],
    axis=1,
)
features = features.astype(float)

print(f"Feature matrix shape: {features.shape}")
features.head()


We now have **10 features** per customer: `Recency`, `Monetary`, `Quantity`, `AvgItemPrice`,
`Age`, 3 one-hot category flags, and 2 one-hot gender flags. Notice the scales already look suspicious —
`Monetary` ranges into the thousands, `Age` ranges 18–64, and the one-hot columns are just 0/1. That's our
cue for the next section.


## 5. Feature Scaling — Why K-Means Cannot Live Without It ⚖️

K-Means (and DBSCAN, and almost every distance-based algorithm) decides which points belong together using
**Euclidean distance**:

$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

The problem: this formula has **no idea** that `Monetary` is measured in dollars (range ~0–2000) while
`Gender_Male` is just 0 or 1. Whichever feature has the *largest numeric range* will completely dominate the
distance calculation — not because it's more *important*, but purely because of its **units**.

**`StandardScaler`** fixes this by transforming every feature to have **mean = 0** and **standard deviation =
1**:

$$z_i = \frac{x_i - \mu_i}{\sigma_i}$$

After scaling, a $50 difference in spending and a 0.5-standard-deviation difference in age are treated as
*comparably meaningful* — which is what we actually want.


In [ ]:
print("Feature ranges BEFORE scaling:")
print(features.agg(["min", "max", "mean", "std"]).T.round(2))


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled = pd.DataFrame(X_scaled, columns=features.columns, index=features.index)

print("Feature ranges AFTER StandardScaler:")
print(X_scaled.agg(["min", "max", "mean", "std"]).T.round(2))


In [ ]:
# Global variables
N_CLUSTERS = # Write your code here (How many groups do you want? Choose wisely 👀)
RANDOM_STATE = # Write your lucky number here (Your lucky ML number ✨)

# Demonstration: cluster the SAME data, unscaled vs scaled, and see how different the result is
km_unscaled = KMeans(
    n_clusters=N_CLUSTERS,
    init="k-means++",
    n_init=10,
    random_state=RANDOM_STATE
)
labels_unscaled = km_unscaled.fit_predict(features)

km_scaled = KMeans(
    n_clusters=N_CLUSTERS,
    init="k-means++",
    n_init=10,
    random_state=RANDOM_STATE
)
labels_scaled = km_scaled.fit_predict(X_scaled)

comparison = pd.DataFrame({
    "Metric": ["Silhouette Score", "Cluster size std-dev (lower = more balanced)"],
    "Unscaled features": [
        round(silhouette_score(features, labels_unscaled), 3),
        round(pd.Series(labels_unscaled).value_counts().std(), 1),
    ],
    "Scaled features": [
        round(silhouette_score(X_scaled, labels_scaled), 3),
        round(pd.Series(labels_scaled).value_counts().std(), 1),
    ],
})
comparison


Look at the cluster sizes in each case (run `pd.Series(labels_unscaled).value_counts()` yourself!). On
unscaled data, K-Means is effectively **clustering almost entirely on `Monetary`** because it has by far the
largest range — Age, category, and gender barely get a vote. On scaled data, every feature contributes fairly.
**Lesson:** always scale before distance-based clustering — this is not a stylistic preference, it silently
changes *which customers get grouped together*.


## 6. The Curse of Dimensionality 🌀

As we add more features (more one-hot columns, more engineered signals), something strange happens to
*distance itself*. In very high dimensions, **all points start to look roughly equally far apart** — the
concept of "near" and "far" that clustering depends on starts to break down.

Let's not just take this on faith — let's **measure it**. We'll generate random points in spaces of
increasing dimensionality and track:

$$\text{Contrast} = \frac{d_{max} - d_{min}}{d_{min}}$$

where $d_{max}$ and $d_{min}$ are the farthest and nearest pairwise distances among the points. A **high**
contrast means "near" and "far" are meaningfully different (good for clustering). A contrast **near zero**
means every point is about the same distance from every other point (bad for clustering — there is no
signal left for a distance-based algorithm to find).


In [ ]:
dims_to_test = [2, 5, 10, 20, 50, 100, 200, 500]
n_points = 200
contrasts = []

for d in dims_to_test:
    rng = np.random.RandomState(RANDOM_STATE)
    pts = rng.uniform(low=0, high=1, size=(n_points, d))
    dists = pdist(pts, metric="euclidean")
    contrast = (dists.max() - dists.min()) / dists.min()
    contrasts.append(contrast)

curse_df = pd.DataFrame({"Dimensions": dims_to_test, "Distance Contrast": contrasts})

plt.figure(figsize=(7, 4.5))
plt.plot(curse_df["Dimensions"], curse_df["Distance Contrast"], marker="o", color="#8172B2")
plt.xscale("log")
plt.xlabel("Number of Dimensions (log scale)")
plt.ylabel("(max dist - min dist) / min dist")
plt.title("The Curse of Dimensionality: Distances Collapse as Dimensions Grow")
plt.show()

curse_df


**What just happened?** With only 2 random dimensions, some points are dramatically closer together
than others — tons of contrast, tons of signal. By 200–500 random dimensions, the *nearest* and *farthest*
points are barely different in distance anymore — the contrast collapses toward zero. Euclidean distance
stops being informative.

**Why this matters for our retail features:** our engineered feature set only has 10 dimensions, so we are
in no real danger yet. But imagine a more realistic production system: one-hot encoding hundreds of product
SKUs, dozens of store locations, and browsing-history embeddings could easily push you into the hundreds or
thousands of dimensions — exactly the regime where raw K-Means/DBSCAN on unreduced features starts to
struggle. That's precisely the problem PCA (next section) is designed to solve.


## 7. PCA Before Clustering — Reduce Dimensions *and* See Your Data 🔭

**Principal Component Analysis (PCA)** finds new axes (principal components) that capture the maximum
possible variance in the data, ordered from most-important to least-important. Using PCA before clustering
gives us two big wins:

1. **Fights the curse of dimensionality** — we keep only the components that carry real signal and drop the
   noisy, low-variance ones.
2. **Free visualization** — even if we clustered on 10 (or 100) dimensions, we can always *look* at the
   result by projecting onto the first 2 principal components.

PCA must be run on **scaled** data — otherwise, just like K-Means, it will be dominated by whichever raw
feature has the largest numeric range.


In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(range(1, len(explained) + 1), explained, color="#4C72B0")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, marker="o", color="#C44E52")
axes[1].axhline(0.90, color="gray", linestyle="--", label="90% variance")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

n_components_90 = int(np.argmax(cumulative >= 0.90) + 1)
print(f"Components needed to explain 90% of variance: {n_components_90}")


In [ ]:
N_PCA_COMPONENTS = # Write your code here (How many dimensions should we keep? 👀)

pca_2d = PCA(
    n_components=N_PCA_COMPONENTS,
    random_state=RANDOM_STATE
)

X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(7, 5.5))
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], alpha=0.5, s=25, color="#55A868")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
plt.title("Customers Projected onto the First 2 Principal Components")
plt.show()


Even before running a single clustering algorithm, this 2D projection already hints at structure — you
can probably squint and see a few loose groupings. That's a great sign: it means our features actually carry
clusterable signal, and it gives us a "ground truth to the eye" that we can sanity-check every algorithm's
output against for the rest of the notebook.


## 8. Choosing the Right Number of Clusters, K 🔢

K-Means requires you to *specify K in advance* — but the business rarely knows in advance whether there are
3 customer segments or 7. We need data-driven ways to choose K. We'll compute **four** complementary metrics
across K = 2..10 on our scaled features:

| Metric | Intuition | Look for |
|---|---|---|
| **Inertia (Elbow Method)** | Sum of squared distances of points to their cluster centroid | The "elbow" where adding more clusters stops helping much |
| **Silhouette Score** | How much closer a point is to its own cluster vs. the next-nearest cluster (range -1 to 1) | **Higher is better** |
| **Davies–Bouldin Index** | Average similarity between each cluster and its most-similar other cluster | **Lower is better** |
| **Calinski–Harabasz Index** | Ratio of between-cluster to within-cluster dispersion | **Higher is better** |

No single metric is perfect — that's *why* we look at all four together.


In [ ]:
k_range = range(2, 11)
inertias, silhouettes, db_scores, ch_scores = [], [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(k_range, inertias, marker="o", color="#4C72B0")
axes[0, 0].set_title("Elbow Method (Inertia) — lower is tighter, watch for the elbow")
axes[0, 0].set_xlabel("K"); axes[0, 0].set_ylabel("Inertia")

axes[0, 1].plot(k_range, silhouettes, marker="o", color="#55A868")
axes[0, 1].set_title("Silhouette Score — higher is better")
axes[0, 1].set_xlabel("K"); axes[0, 1].set_ylabel("Silhouette")

axes[1, 0].plot(k_range, db_scores, marker="o", color="#C44E52")
axes[1, 0].set_title("Davies-Bouldin Index — lower is better")
axes[1, 0].set_xlabel("K"); axes[1, 0].set_ylabel("DB Index")

axes[1, 1].plot(k_range, ch_scores, marker="o", color="#8172B2")
axes[1, 1].set_title("Calinski-Harabasz Index — higher is better")
axes[1, 1].set_xlabel("K"); axes[1, 1].set_ylabel("CH Index")

plt.tight_layout()
plt.show()

k_metrics = pd.DataFrame({
    "K": list(k_range), "Inertia": inertias, "Silhouette": silhouettes,
    "Davies-Bouldin": db_scores, "Calinski-Harabasz": ch_scores,
}).round(3)
k_metrics


In [ ]:
best_k_silhouette = k_metrics.loc[k_metrics["Silhouette"].idxmax(), "K"]
best_k_db = k_metrics.loc[k_metrics["Davies-Bouldin"].idxmin(), "K"]
best_k_ch = k_metrics.loc[k_metrics["Calinski-Harabasz"].idxmax(), "K"]

print(f"Best K by Silhouette Score:      {int(best_k_silhouette)}")
print(f"Best K by Davies-Bouldin Index:  {int(best_k_db)}")
print(f"Best K by Calinski-Harabasz:     {int(best_k_ch)}")


**Decision — and an honest complication.** In this run, Silhouette and Davies-Bouldin both technically
prefer **K=10**, while Calinski-Harabasz prefers **K=2**. This is a very common, very real situation: the
four metrics do **not** agree, because each one is measuring a slightly different notion of "good
clustering," and none of them know anything about what a marketing team can actually *use*. K=10 would carve
customers into segments too small and too numerous for a human team to design separate campaigns for; K=2
throws away almost all nuance.

We deliberately choose **K = 4** as a business-driven compromise: it's small enough for a marketing team to
build four distinct campaigns around, large enough to preserve meaningful differences between groups, and —
as you'll see in Section 14 — it produces genuinely interpretable personas. **This is the single most
important judgment call in the whole notebook, and it's a judgment call, not a formula.** When your metrics
disagree, that's not a bug in your code — it's a signal that the "right" K depends on what you're going to
*do* with the answer.


## 9. K-Means, Step by Step 🎯

**The algorithm (Lloyd's Algorithm), in plain language:**

1. **Initialize**: place K centroids (either randomly, or smartly via K-Means++, see below)
2. **Assign**: give every point to its **nearest** centroid (by Euclidean distance)
3. **Update**: move each centroid to the **mean** position of all points assigned to it
4. **Repeat** steps 2–3 until centroids stop moving (convergence) or a max iteration count is hit

K-Means is essentially a loop of *"assign → average → assign → average..."* until things settle down. Let's
watch it happen on a small 2D toy example so the mechanics are completely visible.


In [ ]:
# Toy 2D data purely for visualizing the algorithm's mechanics
X_toy, _ = make_blobs(n_samples=180, centers=3, cluster_std=0.9, random_state=RANDOM_STATE)

def lloyds_algorithm_steps(X, k, n_iter=4, seed=0):
    rng = np.random.RandomState(seed)
    centroids = X[rng.choice(len(X), size=k, replace=False)]
    history = [centroids.copy()]
    labels = None
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = dists.argmin(axis=1)
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])
        centroids = new_centroids
        history.append(centroids.copy())
    return history, labels

history, final_labels = lloyds_algorithm_steps(X_toy, k=3, n_iter=4, seed=1)

fig, axes = plt.subplots(1, len(history), figsize=(4 * len(history), 4))
for i, (ax, centroids) in enumerate(zip(axes, history)):
    dists = np.linalg.norm(X_toy[:, None, :] - centroids[None, :, :], axis=2)
    labels = dists.argmin(axis=1)
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=labels, cmap="Set2", s=20, alpha=0.7)
    ax.scatter(centroids[:, 0], centroids[:, 1], c="red", marker="X", s=200,
               edgecolor="black", linewidth=1.5)
    ax.set_title(f"Iteration {i}")
plt.suptitle("K-Means in Action: Centroids (red X) Converging Over Iterations", y=1.05)
plt.tight_layout()
plt.show()


Watch the red X's (centroids) — they jump to a random starting spot at Iteration 0, then creep toward
the true center of each blob as the assign→update loop repeats. By Iteration 3 or 4 they've essentially
stopped moving: **convergence**. This is all K-Means ever does — there is no deeper magic, just repeated
averaging.

### Random Initialization vs. K-Means++ 🎲

The starting centroid positions matter a lot. **Random initialization** just picks K random data points as
starting centroids — sometimes you get lucky, sometimes two centroids start right next to each other and the
algorithm settles into a bad local optimum. **K-Means++** (the scikit-learn default) instead spreads out the
initial centroids on purpose, biasing towards points that are far from already-chosen centroids. This gives
more consistent, usually better, results. Let's prove it empirically over many runs.


In [ ]:
n_trials = 30
inertia_random, inertia_plusplus = [], []

for seed in range(n_trials):
    km_r = KMeans(n_clusters=4, init="random", n_init=1, random_state=seed)
    km_r.fit(X_scaled)
    inertia_random.append(km_r.inertia_)

    km_pp = KMeans(n_clusters=4, init="k-means++", n_init=1, random_state=seed)
    km_pp.fit(X_scaled)
    inertia_plusplus.append(km_pp.inertia_)

init_compare = pd.DataFrame({"Random Init": inertia_random, "K-Means++": inertia_plusplus})

plt.figure(figsize=(6.5, 4.5))
sns.boxplot(data=init_compare, palette=["#C44E52", "#55A868"])
plt.ylabel("Final Inertia (lower is better)")
plt.title(f"Inertia Stability Across {n_trials} Runs: Random Init vs. K-Means++")
plt.show()

print(init_compare.agg(["mean", "std", "min", "max"]).round(1))


**K-Means++ typically produces both a lower *and* more consistent (lower std-dev) inertia** across runs
than plain random initialization — fewer unlucky "bad start" outcomes. This is exactly why `init="k-means++"`
is scikit-learn's default, and why we've been using it (along with `n_init=10`, which reruns the whole
algorithm 10 times and keeps the best result) throughout this notebook.


In [ ]:
# Final K-Means model on the real retail data
K_FINAL = # Write your code here
kmeans_final = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=10, random_state=RANDOM_STATE)
df["KMeans_Cluster"] = kmeans_final.fit_predict(X_scaled)

print("Cluster sizes:")
print(df["KMeans_Cluster"].value_counts().sort_index())
print(f"\nSilhouette Score: {silhouette_score(X_scaled, df['KMeans_Cluster']):.3f}")

plt.figure(figsize=(7, 5.5))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=df["KMeans_Cluster"],
                       cmap="Set2", s=30, alpha=0.75)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"K-Means (K={K_FINAL}) Clusters, Visualized via PCA")
plt.legend(*scatter.legend_elements(), title="Cluster", loc="best")
plt.show()


## 10. DBSCAN, Step by Step 🌐

K-Means assumes clusters are **round (convex) blobs of similar size** — because it's literally built around
distance-to-a-single-center. Real customer data (and especially anything with weird shapes — geographic
data, sensor data, fraud rings) often violates that assumption. **DBSCAN** (Density-Based Spatial Clustering
of Applications with Noise) takes a totally different approach: instead of centroids, it looks at **density**.

**Core vocabulary:**
- **`eps` (ε)**: the radius of the neighborhood around each point
- **`min_samples`**: how many points must be within `eps` of a point for that point to count as "core"
- **Core point**: has at least `min_samples` neighbors within `eps`
- **Border point**: not a core point itself, but falls within `eps` of a core point
- **Noise point**: neither — DBSCAN labels these `-1` and simply **does not force them into any cluster**

DBSCAN grows clusters by chaining together core points that are close to each other, then attaching border
points to whichever core-point-chain they touch. **It never needs to be told K in advance**, and it can find
clusters of *any shape* — but it can also decide some points are just noise, which K-Means can never do.

### Where K-Means fails and DBSCAN shines 🌙


In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=RANDOM_STATE)

km_moons = KMeans(n_clusters=2, init="k-means++", n_init=10, random_state=RANDOM_STATE)
labels_km_moons = km_moons.fit_predict(X_moons)

db_moons = DBSCAN(eps=0.2, min_samples=5)
labels_db_moons = db_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km_moons, cmap="Set2", s=25)
axes[0].set_title("K-Means on Non-Convex Data — FAILS\n(cuts the moons in half)")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db_moons, cmap="Set2", s=25)
axes[1].set_title("DBSCAN on Non-Convex Data — SUCCEEDS\n(follows the true shape)")

plt.tight_layout()
plt.show()


This is the single clearest illustration of *why* the clustering algorithm you pick matters. K-Means,
by construction, can only ever draw **straight-line (linear) boundaries** between clusters — it slices these
two crescents right down the middle, mixing both. DBSCAN follows the actual **density** of the points and
recovers the two crescents perfectly, with zero knowledge that K=2 was even the right number.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, eps in zip(axes, [0.1, 0.2, 0.4]):
    db = DBSCAN(eps=eps, min_samples=5).fit(X_moons)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise = list(db.labels_).count(-1)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=db.labels_, cmap="Set2", s=25)
    ax.set_title(f"eps={eps}\n{n_clusters} clusters, {n_noise} noise pts")
plt.suptitle("Effect of `eps`: Too Small = Everything is Noise, Too Large = Clusters Merge", y=1.05)
plt.tight_layout()
plt.show()


**Too-small `eps`** → almost no point has enough close neighbors, so nearly everything becomes noise.
**Too-large `eps`** → the whole dataset can collapse into a single giant cluster. There's a sweet spot in
between, and we can find it systematically with a **k-distance plot**: for every point, compute the distance
to its `k`-th nearest neighbor (`k = min_samples`), sort those distances, and look for the "elbow" — that
elbow is a great starting value for `eps`.


In [ ]:
min_samples_choice = # Write your code here (How many neighbors should we check? 👀)
neighbors = NearestNeighbors(n_neighbors=min_samples_choice).fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(7, 4.5))
plt.plot(k_distances, color="#4C72B0")
plt.xlabel("Points, sorted by distance")
plt.ylabel(f"Distance to {min_samples_choice}-th nearest neighbor")
plt.title("k-Distance Plot — Look for the 'Elbow' to Pick `eps`")
plt.show()


In [ ]:
# Apply DBSCAN to our real, scaled retail features using an eps read off the elbow above
EPS_CHOICE = # Write your code here (How big should the neighborhood be? 👀)
db_retail = DBSCAN(eps=EPS_CHOICE, min_samples=min_samples_choice)
df["DBSCAN_Cluster"] = db_retail.fit_predict(X_scaled)

n_clusters_db = len(set(df["DBSCAN_Cluster"])) - (1 if -1 in df["DBSCAN_Cluster"].values else 0)
n_noise_db = (df["DBSCAN_Cluster"] == -1).sum()
print(f"DBSCAN found {n_clusters_db} clusters and flagged {n_noise_db} points as noise "
      f"({n_noise_db / len(df):.1%} of the data)")

if n_clusters_db >= 2:
    mask = df["DBSCAN_Cluster"] != -1
    sil_db = silhouette_score(X_scaled[mask], df.loc[mask, "DBSCAN_Cluster"])
    print(f"Silhouette Score (excluding noise): {sil_db:.3f}")

plt.figure(figsize=(7, 5.5))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=df["DBSCAN_Cluster"], cmap="Set2", s=30, alpha=0.75)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"DBSCAN (eps={EPS_CHOICE}, min_samples={min_samples_choice}) on Retail Data")
plt.legend(*scatter.legend_elements(), title="Cluster (-1 = noise)", loc="best")
plt.show()


**Reading this result honestly:** DBSCAN actually found **6 clusters** here, plus only **17 noise points**
(1.7% of the data) — and if you inspect the cluster sizes, they come out almost perfectly even (roughly 140–173
customers each). Why 6? Recall our feature set one-hot-encodes **3 product categories × 2 genders = 6
combinations**. Because one-hot dummy variables sit at the *corners* of a hypercube, each Gender × Category
combination forms its own tight, well-separated density island once the data is scaled — and DBSCAN, which
looks for exactly that kind of density gap, finds all 6 corners cleanly, almost independent of the
continuous spending features.

**This is a genuinely important lesson, not just a quirk:** both DBSCAN *and* K-Means (Section 9) ended up
organizing customers primarily by **category and gender**, not by **spending value**. That's a direct
consequence of a feature-engineering choice — 5 categorical dummy columns vs. 5 continuous columns gave the
categorical structure a very strong vote in a 10-dimensional distance calculation. It's not wrong, but it's
worth noticing *before* you present "customer segments" to a business stakeholder. (This is exactly what
Homework #1 asks you to test — rerun clustering with the dummies removed and see if a value-based
segmentation emerges instead.)


## 11. Two More Tools Worth Knowing 🧰

K-Means and DBSCAN cover most day-to-day clustering needs, but two more algorithms come up often enough that
every ML practitioner should recognize them.

### 11.1 Hierarchical (Agglomerative) Clustering

Instead of picking K upfront, hierarchical clustering starts with **every point as its own cluster**, then
repeatedly merges the two closest clusters until only one giant cluster remains. The whole merge history is
drawn as a **dendrogram** — a tree you can "cut" at any height to get any number of clusters, *after the fact*.
This is extremely useful when you genuinely don't know K and want to *see* the nested structure before
committing.


In [ ]:
# Use a random sub-sample so the dendrogram stays readable
sample_idx = X_scaled.sample(40, random_state=RANDOM_STATE).index
Z = linkage(X_scaled.loc[sample_idx], method="ward")

plt.figure(figsize=(13, 5))
dendrogram(Z, labels=sample_idx.astype(str).tolist(), leaf_rotation=90, leaf_font_size=7)
plt.title("Dendrogram — Agglomerative (Ward) Clustering on a 40-Customer Sample")
plt.xlabel("Customer index")
plt.ylabel("Merge distance")
plt.show()

agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward")
df["Agglomerative_Cluster"] = agg.fit_predict(X_scaled)
print(f"Silhouette Score (Agglomerative, K={K_FINAL}): "
      f"{silhouette_score(X_scaled, df['Agglomerative_Cluster']):.3f}")


### 11.2 Gaussian Mixture Models (GMM) — "Soft" Clustering

K-Means gives every point a single, hard cluster assignment. A **Gaussian Mixture Model** instead assumes the
data was generated by a mixture of several overlapping Gaussian ("bell curve") distributions, and gives every
point a **probability** of belonging to each cluster. This matters a lot in retail: a customer might be
`70% "Occasional Big Spender", 30% "High-Value Loyalist"` — a soft, probabilistic label is often more honest
than forcing a hard choice, and you can flag genuinely ambiguous customers for a human to review.


In [ ]:
gmm = GaussianMixture(n_components=K_FINAL, random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(X_scaled)
gmm_probs = gmm.predict_proba(X_scaled)

df["GMM_Cluster"] = gmm_labels
print(f"Silhouette Score (GMM, K={K_FINAL}): {silhouette_score(X_scaled, gmm_labels):.3f}")

# Show a few genuinely "ambiguous" customers: no single cluster has > 60% probability
max_prob = gmm_probs.max(axis=1)
ambiguous = np.argsort(max_prob)[:5]
print("\n5 most ambiguous customers (no dominant cluster probability):")
pd.DataFrame(gmm_probs[ambiguous], columns=[f"P(Cluster {i})" for i in range(K_FINAL)]).round(2)


### 11.3 Deep-Learning-Based Clustering (Autoencoder + K-Means)

Our decision guide (Section 12) has a rule of thumb: **when should we go deeper than classic K-Means/DBSCAN
and reach for deep learning?** Answer: when the *raw* feature space is high-dimensional and messy in a way
tabular scaling/PCA can't fully untangle — think raw images, free-text reviews, clickstream sequences, or
hundreds of noisy engineered columns. The standard recipe is:

1. Train an **autoencoder** — a neural network that compresses the input down to a small "bottleneck" layer
   and then tries to reconstruct the original input from that bottleneck
2. Once trained, throw away the decoder and keep only the **encoder** — its bottleneck output is a compact,
   learned, non-linear embedding of each data point
3. Run K-Means (or DBSCAN, or GMM) on those **embeddings** instead of the raw features

Our retail feature set is tabular and only 10-dimensional, so classic PCA + K-Means is already the right,
simpler tool (see Section 12!). But it's important to *see* the autoencoder pattern in code once, since it's
the technique you'll reach for the moment your features stop being simple numeric columns.


In [ ]:
from sklearn.neural_network import MLPRegressor

# A tiny "autoencoder" via a bottleneck MLPRegressor trained to reconstruct its own input.
# This is a deliberately lightweight, CPU-friendly stand-in for a PyTorch/Keras autoencoder,
# so the whole notebook still runs in seconds on Colab's free tier.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(8, 2, 8),   # 10 -> 8 -> 2 (bottleneck) -> 8 -> 10
    activation="tanh",
    max_iter=2000,
    random_state=RANDOM_STATE,
)
autoencoder.fit(X_scaled, X_scaled)
reconstruction_error = np.mean((autoencoder.predict(X_scaled) - X_scaled.values) ** 2)
print(f"Autoencoder reconstruction MSE: {reconstruction_error:.4f}")

# Extract the learned 2-neuron bottleneck as our "deep embedding"
W1 = autoencoder.coefs_[0]; b1 = autoencoder.intercepts_[0]
W2 = autoencoder.coefs_[1]; b2 = autoencoder.intercepts_[1]
hidden1 = np.tanh(X_scaled.values @ W1 + b1)
bottleneck = np.tanh(hidden1 @ W2 + b2)

kmeans_deep = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=10, random_state=RANDOM_STATE)
deep_labels = kmeans_deep.fit_predict(bottleneck)
print(f"Silhouette Score (K-Means on autoencoder embedding): "
      f"{silhouette_score(bottleneck, deep_labels):.3f}")

plt.figure(figsize=(7, 5.5))
plt.scatter(bottleneck[:, 0], bottleneck[:, 1], c=deep_labels, cmap="Set2", s=30, alpha=0.75)
plt.xlabel("Learned dimension 1"); plt.ylabel("Learned dimension 2")
plt.title("K-Means on a Deep Autoencoder's 2D Bottleneck Embedding")
plt.show()


Notice the deep-learning route needed **far more code, more hyperparameters, and a longer training
step** — for a silhouette score that's usually roughly comparable to plain PCA + K-Means on this dataset. This
is the lesson, not a footnote: **deep learning earns its complexity when the raw features are genuinely
complex (images, text, sequences).** For clean tabular data like ours, PCA + K-Means is simpler, faster, more
interpretable, *and* just as good. Reach for the bigger hammer only when the nail actually needs it.


## 12. Decision Guide: K-Means vs. DBSCAN vs. Deep-Learning-Based 🧭

| Question | Choose... |
|---|---|
| Do you roughly know how many segments you expect, and are they likely to be round/blob-shaped? | **K-Means** |
| Do you *not* know K, suspect weird/non-convex shapes, or need automatic outlier/noise detection? | **DBSCAN** |
| Do all your features fit neatly in a table with < ~50-ish meaningful dimensions after PCA? | **K-Means or DBSCAN on PCA-reduced features** |
| Is your raw data images, free text, audio, clickstreams, or otherwise not naturally tabular/numeric? | **Deep-learning-based (autoencoder / embeddings) + K-Means or DBSCAN on top** |
| Do you want a soft, probabilistic cluster membership instead of a hard assignment? | **Gaussian Mixture Model** |
| Do you want to explore *several* possible values of K visually before committing to one? | **Hierarchical Clustering (dendrogram)** |
| Is interpretability for a non-technical business stakeholder the top priority? | **K-Means** (centroids are easy to describe as "the average customer in this group") |
| Are your engineered features mostly one-hot categorical dummies at low cardinality? | **DBSCAN often does surprisingly well** — one-hot corners *are* density islands |

**For our retail dataset specifically:** as Section 13 will show numerically, **DBSCAN actually edges out
K-Means on Silhouette and Davies-Bouldin** in this run — because, as Section 10 explained, our feature space
is effectively a handful of discrete Gender × Category corners, which is exactly the kind of density
structure DBSCAN is built to find. **This is a useful, humbling result**: the Decision Guide table above is a
starting heuristic, not a guarantee — the only way to know which algorithm truly wins on *your* data is to
compute the metrics, as we're about to do in Section 13, rather than assume it from theory alone. K-Means
still wins on *interpretability* (its centroids translate directly into "the average customer in this group"),
which is why we carried it forward as the basis for the personas in Section 14 — but a sharp practitioner
would flag DBSCAN's stronger numbers and consider building personas from *its* 6 clusters instead.


## 13. Cluster Validation — Comparing All Methods Side by Side 📋

Let's put every algorithm we ran on the *same* real retail feature set into one summary table, so we can
compare them on equal footing using our three core validation metrics.


In [ ]:
def safe_metric(metric_fn, X, labels):
    mask = labels != -1  # exclude DBSCAN noise points from validation metrics
    if len(set(labels[mask])) < 2:
        return np.nan
    X_masked = X.loc[mask] if isinstance(X, pd.DataFrame) else X[mask]
    return metric_fn(X_masked, labels[mask])

results = []
for name, labels in [
    ("K-Means", df["KMeans_Cluster"].values),
    ("DBSCAN", df["DBSCAN_Cluster"].values),
    ("Agglomerative", df["Agglomerative_Cluster"].values),
    ("Gaussian Mixture", df["GMM_Cluster"].values),
]:
    results.append({
        "Algorithm": name,
        "# Clusters found": len(set(labels)) - (1 if -1 in labels else 0),
        "# Noise points": int((labels == -1).sum()),
        "Silhouette (higher better)": round(safe_metric(silhouette_score, X_scaled, labels), 3),
        "Davies-Bouldin (lower better)": round(safe_metric(davies_bouldin_score, X_scaled, labels), 3),
        "Calinski-Harabasz (higher better)": round(safe_metric(calinski_harabasz_score, X_scaled, labels), 1),
    })

validation_summary = pd.DataFrame(results)
validation_summary


**DBSCAN actually posts the best Silhouette (0.294) and Davies-Bouldin (1.500) scores here** — a good
reminder that the Decision Guide's *default* recommendation (K-Means for clean tabular data) is a starting
heuristic, not a rule. K-Means and Agglomerative Clustering (both K=4) come in essentially tied just behind
it, and Gaussian Mixture trails slightly. Two things are both true at once: **DBSCAN found a numerically
tighter, more separated clustering** (because it's matching the natural Gender × Category corners almost
exactly), while **K-Means gives us the simpler K=4 result** that's easier to hand to a non-technical
marketing stakeholder as four named personas. In your own work, when a "worse on paper" algorithm is
dramatically easier to explain and act on, that's a completely legitimate reason to prefer it — but you
should always know the trade-off you're making, and say so explicitly, rather than silently picking the
result you like best.


## 14. From Clusters to Customer Personas 👥

This is the step that turns "we ran an algorithm" into "we changed the business." We'll use our **K-Means
(K=4)** result — the winner from Section 13 — and profile each cluster on every feature that matters to a
marketing team: recency, spend, basket size, item price point, age, category preference, and gender split.


In [ ]:
profile = features.copy()
profile["Cluster"] = df["KMeans_Cluster"]
profile["Gender"] = df["Gender"].values
profile["Product Category"] = df["Product Category"].values

cluster_profile = profile.groupby("Cluster").agg(
    Num_Customers=("Monetary", "size"),
    Avg_Recency_Days=("Recency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
    Avg_Quantity=("Quantity", "mean"),
    Avg_ItemPrice=("AvgItemPrice", "mean"),
    Avg_Age=("Age", "mean"),
).round(1)

cluster_profile["Pct_of_Customers"] = (cluster_profile["Num_Customers"] / len(profile) * 100).round(1)
cluster_profile["Pct_of_Revenue"] = (cluster_profile["Total_Revenue"] / profile["Monetary"].sum() * 100).round(1)
cluster_profile = cluster_profile.sort_values("Avg_Monetary", ascending=False)
cluster_profile


In [ ]:
top_category = profile.groupby("Cluster")["Product Category"].agg(lambda s: s.value_counts().idxmax())
gender_split = profile.groupby("Cluster")["Gender"].agg(lambda s: s.value_counts(normalize=True).round(2).to_dict())

print("Most common category per cluster:")
print(top_category)
print("\nGender split per cluster:")
for c, d in gender_split.items():
    print(f"  Cluster {c}: {d}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = cluster_profile.index
axes[0].bar(order.astype(str), cluster_profile.loc[order, "Avg_Monetary"], color="#4C72B0")
axes[0].set_title("Average Spend per Customer, by Cluster")
axes[0].set_xlabel("Cluster"); axes[0].set_ylabel("Average Monetary ($)")

axes[1].bar(cluster_profile.loc[order, "Pct_of_Customers"].index.astype(str),
            cluster_profile.loc[order, "Pct_of_Customers"], label="% of Customers", alpha=0.7, color="#8172B2")
axes[1].bar(cluster_profile.loc[order, "Pct_of_Revenue"].index.astype(str),
            cluster_profile.loc[order, "Pct_of_Revenue"], label="% of Revenue", alpha=0.5, color="#C44E52",
            width=0.5)
axes[1].set_title("Share of Customers vs. Share of Revenue, by Cluster")
axes[1].set_xlabel("Cluster"); axes[1].legend()

plt.tight_layout()
plt.show()


### 🏷️ Naming the personas — reading the real output

Here's what `cluster_profile`, the top-category table, and the gender-split actually showed for our K=4
run:

| Cluster | Size | % of Customers | % of Revenue | Dominant Category | Gender | Avg. Recency (days) |
|---|---|---|---|---|---|---|
| 0 | 141 | 14.1% | 15.1% | Beauty | 100% Male | 178.6 |
| 3 | 166 | 16.6% | 16.4% | Beauty | 100% Female | 188.1 |
| 1 | 351 | 35.1% | 34.1% | Clothing | Mixed | **194.9 (highest)** |
| 2 | 342 | 34.2% | 34.4% | Electronics | Mixed | 170.0 |

**First honest observation:** `Pct_of_Customers` and `Pct_of_Revenue` are nearly identical for every single
cluster. There is **no strong 80/20 Pareto skew** in this run — average spend per customer barely differs
across clusters (roughly $443–$487). As Section 10 explained, that's because these clusters are mainly
separated by **category and gender**, not by spending tier. That's a real, useful finding on its own: in
*this* dataset, "who buys what" is a much stronger organizing signal than "who spends the most."

**Turning that into four usable personas anyway:**

| Persona | Cluster | Marketing action | Business impact |
|---|---|---|---|
| **🧔 Male Beauty Shoppers** | 0 | Men's grooming is typically under-marketed — bundle grooming kits, target with male-specific beauty content/ads | Grow basket size in a niche, currently-smaller (14.1% of customers) but full-price segment |
| **💄 Female Beauty Shoppers** | 3 | Beauty is a naturally repeat-purchase category — launch a replenishment subscription or loyalty points program | Beauty products run out — a subscription converts one-off buyers into recurring revenue |
| **👕 Clothing Shoppers (Mixed Gender)** | 1 | This is our **largest cluster (35.1% of customers)** *and* has the **highest average recency (194.9 days)** — i.e. our best win-back target. Send a lapsed-customer discount code for new seasonal arrivals | Section 15 estimates the concrete revenue recoverable by acting on exactly this signal |
| **📱 Electronics Shoppers (Mixed Gender)** | 2 | Highest `AvgItemPrice` category and **most recent average purchases (170 days)** — this segment is currently engaged; upsell extended warranties/accessories and time offers around new product launches | Protect and grow the segment already closest to a repeat purchase, rather than only chasing lapsed customers |

The takeaway: even without a dramatic value-based Pareto split, clustering still gave us four **operationally
distinct** groups — different favorite category, different gender mix, and (crucially) different recency —
which is more than enough to justify four different campaigns instead of one generic newsletter to everybody.


## 15. Turning Personas into a Profit Estimate 💵

A cluster profile is only useful once someone can answer: *"okay, so what happens to our revenue if we act on
this?"* Let's build one concrete, transparent example: a **win-back campaign for the At-Risk / high-recency
segment**. We'll clearly label every assumption so you can swap in your own numbers for a real campaign.


In [ ]:
# Identify the highest-recency (longest since purchase) cluster as our "at-risk" target
at_risk_cluster = cluster_profile["Avg_Recency_Days"].idxmax()
at_risk_customers = cluster_profile.loc[at_risk_cluster, "Num_Customers"]
at_risk_avg_spend = cluster_profile.loc[at_risk_cluster, "Avg_Monetary"]

# --- Assumptions (label these clearly -- swap in real historical campaign data when you have it) ---
ASSUMED_WINBACK_RESPONSE_RATE = 0.15   # 15% of targeted at-risk customers make a repeat purchase
ASSUMED_REPEAT_SPEND_RATIO = 0.8       # a win-back purchase tends to be ~80% of their original spend

expected_reactivations = at_risk_customers * ASSUMED_WINBACK_RESPONSE_RATE
expected_recovered_revenue = expected_reactivations * at_risk_avg_spend * ASSUMED_REPEAT_SPEND_RATIO

print(f"Target segment: Cluster {at_risk_cluster} (highest average recency = "
      f"{cluster_profile.loc[at_risk_cluster, 'Avg_Recency_Days']:.0f} days since last purchase)")
print(f"Customers in segment: {at_risk_customers}")
print(f"Assumed response rate: {ASSUMED_WINBACK_RESPONSE_RATE:.0%}")
print(f"Expected reactivated customers: {expected_reactivations:.1f}")
print(f"Expected recovered revenue: ${expected_recovered_revenue:,.0f}")
print(f"\nAs a share of this dataset's total revenue: "
      f"{expected_recovered_revenue / profile['Monetary'].sum():.2%}")


This is exactly the kind of one-page number a marketing team needs to greenlight a campaign — and
notice it required **every technique from this notebook**: scaling, PCA, choosing K, K-Means, and honest
cluster profiling, all in service of one clean, defensible business estimate. Change the assumed response
rate or repeat-spend ratio to run your own "what-if" scenarios — that's the whole point of making the
assumptions explicit variables instead of hiding them inside a single hardcoded number.


## 16. Key Takeaways 🎓

1. **Unsupervised learning finds structure with no labels** — the payoff is entirely in how well you turn
   that structure into a business story and an action (Sections 1, 14, 15)
2. **Feature scaling is mandatory, not optional**, for any distance-based algorithm — an unscaled `Monetary`
   column will silently hijack your entire clustering result (Section 5)
3. **The Curse of Dimensionality is real and measurable** — distances collapse as dimensions grow, which is
   exactly why PCA earns its place in almost every clustering pipeline (Sections 6–7)
4. **Choosing K is a multi-metric decision** — Elbow, Silhouette, Davies-Bouldin, and Calinski-Harabasz each
   tell a slightly different part of the story; let business interpretability break ties (Section 8)
5. **K-Means++ beats random initialization** in both average quality and run-to-run consistency — always
   prefer it over plain random init (Section 9)
6. **DBSCAN can do things K-Means fundamentally cannot**: find non-convex shapes and flag genuine outliers as
   noise, at the cost of needing careful `eps`/`min_samples` tuning (Section 10)
7. **Hierarchical clustering and GMMs** round out the toolkit — dendrograms for exploring K visually, GMMs for
   honest, probabilistic ("soft") cluster membership (Section 11)
8. **Deep-learning-based clustering (autoencoders) is a tool for genuinely high-dimensional, unstructured
   data** — not a default upgrade over PCA + K-Means on clean tabular features (Sections 11, 12)
9. The single most important, most-often-skipped step: **translate cluster centroids into customer personas,
   and personas into a dollar-and-cents action plan** (Sections 14–15)

### Final validation leaderboard (recap)


In [ ]:
validation_summary.sort_values("Silhouette (higher better)", ascending=False, na_position="last")


---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 5 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>